In [1]:
from dotenv import load_dotenv
load_dotenv()  

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=1.0,
    max_retries=0,
)

### Dicuss the the problem

In [3]:
text = "Hello, my name is Ayush, Email is ayush@gmail.com and my age is 21"

In [4]:
res = llm.invoke(f"Please give me only name email and age from this test: {text}")

/Users/ayushsinghchandel/Desktop/GenAI/env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [5]:
print(res.content)

[{'type': 'text', 'text': '**Name:** Ayush  \n**Email:** ayush@gmail.com  \n**Age:** 21', 'extras': {'signature': 'EscHCsQHAWkUfRPZ5MqxOnJgSHdTzd8WmLjYEun/Wtj9SAWpNfHH0Gpng1zcjS19EPvO92QGBDX82kmYacDprPFT65F6Uc7ZTfCC2/bPxo8KZTNh0xIrcML3aG2G5XdnDxuB90KmL9/Q/KHCaVjGjxdp1OC7JJ6mlVmUe+I4EHCvEYQQnRiGwBzKsoQJP61eNOhG2JXDV67WQ2GJKHkg5KuDlsqFWBqfK5BpPu/b7wC4Kp8q+woxHCEWAtS3IQNk8HWZCDdY1oOVSsCafTJYnGPHT9ar/d9Ygpk194BeI0+9NlQWNj0+o7r/erQcQOj/BJRO6DqYnjaqAiJt/KxVp/YPaI0anBujwcmsA81KTOWCT0olpIzLYj4+Fh1BtahRAZKtfEI70EzDG2kZKtw1FyExPYbNNLvq0Tlt2fbA1aG6+ObxfxxUWrIP6cLYn0PW9ai8u96mSy8r9sWgGwslcUeiYHP2rJK221Oowzpb9ehbHQHx40MVrlBU/A5+O+5jTzXXMPJlQih/ZemAgNj7w1iSV+6XLIS7qyODwzCBYerOzHTzTW22Hy6GDIiM9tJajti9N9AmIeALSNYft2wdr+9z9NKZ5ebBuhGqgYVpPTRuxwH8So5oAhhVru9AbfLpF84ZuptxBZSHLDEqpCfvtQVCm7o9PUm7YhaxhK30KtEy63T0GFGC2hpJpcK0JX2dfj5HvPmL3N72baSmLOqyIBWjI57tzv/t7I248NSXxbGS2JTbPtDESrUM+e7o9Il/m57RLhNNyhWvxRvi9+v/U2j9pZ52XtLndv26e27ucDdbbOtm6ydNFyVn3/0FkU+Wb9fbPj32EpUb5A0XF1jEdDBpdGaCjd34un2HIOfDUV8TRhJbyyS1i

### Solution is: Structured Output

In [25]:
from pydantic import BaseModel, Field
from typing import List
class ResponseStructure(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    age: int = Field(description="The age of the person")
structured_llm = llm.with_structured_output(ResponseStructure)

In [12]:
res = structured_llm.invoke(f"Please give me only name email and age from this test: {text}")
print(res)

/Users/ayushsinghchandel/Desktop/GenAI/env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


name='Ayush' email='ayush@gmail.com' age=21


In [13]:
res

ResponseStructure(name='Ayush', email='ayush@gmail.com', age=21)

In [14]:
res.name

'Ayush'

In [15]:
res.age

21

In [16]:
res.email

'ayush@gmail.com'

In [17]:
res.model_dump()

{'name': 'Ayush', 'email': 'ayush@gmail.com', 'age': 21}

In [18]:
class Movies(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

structured_llm_movies = llm.with_structured_output(Movies)

In [20]:
structured_llm_movies.invoke("Please give me one lastest sci-fi movie")

/Users/ayushsinghchandel/Desktop/GenAI/env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Movies(title='Dune: Part Two', year=2024, genre='Sci-Fi', director='Denis Villeneuve', rating=8.6)

In [28]:
class AllMovies(BaseModel):
    movies: List[Movies]

movies_llm = llm.with_structured_output(AllMovies)

In [29]:
movies_llm.invoke("Please give me 3 lastest sci-fi movies")

/Users/ayushsinghchandel/Desktop/GenAI/env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AllMovies(movies=[Movies(title='Dune: Part Two', year=2024, genre='Sci-Fi', director='Denis Villeneuve', rating=8.6), Movies(title='Kingdom of the Planet of the Apes', year=2024, genre='Sci-Fi', director='Wes Ball', rating=7.0), Movies(title='Furiosa: A Mad Max Saga', year=2024, genre='Sci-Fi', director='George Miller', rating=7.6)])